# Text Classification: Sentiment Analysis

This notebook builds a complete text classification pipeline:
1. Load and explore a **sentiment analysis** dataset
2. Feature extraction with **TF-IDF**
3. Train **Naive Bayes**, **Logistic Regression**, and **SVM** classifiers
4. Evaluate and compare performance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline

%matplotlib inline

## 1. Load Dataset

We use a subset of the **20 Newsgroups** dataset, selecting 4 categories to simulate a sentiment/topic classification task.

In [ ]:
categories = ['sci.med', 'sci.space', 'rec.sport.baseball', 'talk.politics.mideast']

newsgroups = fetch_20newsgroups(subset='all', categories=categories,
                                remove=('headers', 'footers', 'quotes'),
                                random_state=42)

X_text = newsgroups.data
y = newsgroups.target
target_names = newsgroups.target_names

print(f"Documents: {len(X_text)}")
print(f"Classes: {target_names}")
print(f"Class distribution: {np.bincount(y)}")
print(f"\nSample document (first 200 chars):\n{X_text[0][:200]}")

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## 2. Feature Extraction: TF-IDF

We convert text to numerical features using TF-IDF with n-grams.

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF shape: {X_train_tfidf.shape}")
print(f"Sparsity: {1 - X_train_tfidf.nnz / np.prod(X_train_tfidf.shape):.2%}")

## 3. Train Classifiers

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0),
    'Linear SVM': LinearSVC(max_iter=2000, C=1.0),
}

results = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = np.mean(y_pred == y_test)
    results[name] = {'model': model, 'predictions': y_pred, 'accuracy': acc}
    print(f"{name:25s} accuracy: {acc:.4f}")

In [ ]:
# Detailed report for best model
best_name = max(results, key=lambda k: results[k]['accuracy'])
best = results[best_name]
print(f"Best model: {best_name}\n")
print(classification_report(y_test, best['predictions'], target_names=target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, best['predictions'])
disp = ConfusionMatrixDisplay(cm, display_labels=[n.split('.')[-1] for n in target_names])
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues')
plt.title(f'Confusion Matrix -- {best_name}')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation comparison
fig, ax = plt.subplots(figsize=(8, 5))
cv_results = {}
for name in models:
    pipe = make_pipeline(
        TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english'),
        models[name].__class__(**models[name].get_params())
    )
    scores = cross_val_score(pipe, X_text, y, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:25s} CV: {scores.mean():.4f} +/- {scores.std():.4f}")

ax.boxplot(cv_results.values(), labels=cv_results.keys())
ax.set_ylabel('Accuracy')
ax.set_title('5-Fold Cross-Validation')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Most Informative Features

Examining which words the model relies on most.

In [ ]:
# For Logistic Regression: top features per class
lr_model = results['Logistic Regression']['model']
feature_names = tfidf.get_feature_names_out()

for i, cat in enumerate(target_names):
    top_idx = lr_model.coef_[i].argsort()[-8:][::-1]
    top_words = [feature_names[j] for j in top_idx]
    print(f"{cat}: {', '.join(top_words)}")

## Key Takeaways

- **TF-IDF + linear models** is a strong baseline for text classification.
- **Logistic Regression** and **Linear SVM** typically outperform Naive Bayes on larger vocabularies.
- Feature inspection reveals which words drive predictions -- important for interpretability.

**Next:** Transformer models with Hugging Face.